In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

In [2]:
encoded_sequences = np.load(
    "../encoded_sequences_len60_v2.npy"
)

print(
    encoded_sequences.shape
)

(995, 62)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(
    encoded_sequences,
    test_size=0.1,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(895, 62)
(100, 62)


In [4]:
import torch

X_train = torch.tensor(
    X_train,
    dtype=torch.long
)

X_test = torch.tensor(
    X_test,
    dtype=torch.long
)

In [8]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train)
test_dataset = TensorDataset(X_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print(len(train_loader))

28


In [9]:
print(X_train.shape)
print(X_test.shape)
print(len(train_loader))

torch.Size([895, 62])
torch.Size([100, 62])
28


In [10]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=62
    ):
        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            *
            (
                -math.log(10000.0)
                / d_model
            )
        )

        pe[:,0::2] = torch.sin(
            position * div_term
        )

        pe[:,1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self,x):

        return (
            x
            +
            self.pe[
                :,
                :x.size(1)
            ]
        )

In [11]:
import torch
import torch.nn as nn

class TransformerVAE(nn.Module):

    def __init__(
        self,
        vocab_size=23,
        embed_dim=128,
        latent_dim=64,
        max_len=62
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.pos_encoder = PositionalEncoding(
            embed_dim,
            max_len
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=8,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=3
        )

        self.fc_mu = nn.Linear(
            embed_dim,
            latent_dim
        )

        self.fc_logvar = nn.Linear(
            embed_dim,
            latent_dim
        )

        self.latent_to_embed = nn.Linear(
            latent_dim,
            embed_dim
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=8,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=3
        )

        self.output_layer = nn.Linear(
            embed_dim,
            vocab_size
        )

    def encode(self,x):

        x = self.embedding(x)

        x = self.pos_encoder(x)

        enc = self.encoder(x)

        pooled = enc.mean(dim=1)

        mu = self.fc_mu(pooled)

        logvar = self.fc_logvar(pooled)

        return mu, logvar

    def reparameterize(
        self,
        mu,
        logvar
    ):

        std = torch.exp(
            0.5 * logvar
        )

        eps = torch.randn_like(
            std
        )

        return mu + eps * std

    def decode(
        self,
        z,
        decoder_input
    ):

        tgt = self.embedding(
            decoder_input
        )

        tgt = self.pos_encoder(
            tgt
        )

        memory = self.latent_to_embed(
            z
        ).unsqueeze(1)

        out = self.decoder(
            tgt,
            memory
        )

        logits = self.output_layer(
            out
        )

        return logits

    def forward(self,x):

        mu, logvar = self.encode(x)

        z = self.reparameterize(
            mu,
            logvar
        )

        decoder_input = x[:,:-1]

        logits = self.decode(
            z,
            decoder_input
        )

        return logits, mu, logvar

In [12]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

vae = TransformerVAE(
    max_len=62
).to(device)

print(
    sum(
        p.numel()
        for p in vae.parameters()
    )
)

3787799


In [13]:
import torch.nn.functional as F

def vae_loss(
    logits,
    target,
    mu,
    logvar,
    beta=0.01
):

    recon_loss = F.cross_entropy(
        logits.reshape(-1,23),
        target.reshape(-1),
        ignore_index=0
    )

    kl_loss = -0.5 * torch.mean(
        1 +
        logvar -
        mu.pow(2) -
        logvar.exp()
    )

    total_loss = (
        recon_loss
        +
        beta * kl_loss
    )

    return (
        total_loss,
        recon_loss,
        kl_loss
    )

In [14]:
optimizer = torch.optim.Adam(
    vae.parameters(),
    lr=1e-4
)

In [15]:
num_epochs = 30

for epoch in range(num_epochs):

    vae.train()

    total_loss = 0

    for batch in train_loader:

        x = batch[0].to(device)

        logits, mu, logvar = vae(x)

        target = x[:,1:]

        loss, recon, kl = vae_loss(
            logits,
            target,
            mu,
            logvar
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss={loss:.4f} "
        f"Recon={recon:.4f} "
        f"KL={kl:.4f}"
    )

Epoch [1/30] Loss=2.9233 Recon=2.9203 KL=0.3036
Epoch [2/30] Loss=2.8159 Recon=2.8146 KL=0.1276
Epoch [3/30] Loss=2.8104 Recon=2.8086 KL=0.1854
Epoch [4/30] Loss=2.6544 Recon=2.6523 KL=0.2142
Epoch [5/30] Loss=2.6710 Recon=2.6683 KL=0.2718
Epoch [6/30] Loss=2.6684 Recon=2.6654 KL=0.3061
Epoch [7/30] Loss=2.5827 Recon=2.5789 KL=0.3723
Epoch [8/30] Loss=2.5568 Recon=2.5526 KL=0.4167
Epoch [9/30] Loss=2.5448 Recon=2.5400 KL=0.4790
Epoch [10/30] Loss=2.4722 Recon=2.4667 KL=0.5502
Epoch [11/30] Loss=2.3881 Recon=2.3820 KL=0.6124
Epoch [12/30] Loss=2.4098 Recon=2.4029 KL=0.6875
Epoch [13/30] Loss=2.2604 Recon=2.2526 KL=0.7773
Epoch [14/30] Loss=2.1225 Recon=2.1139 KL=0.8591
Epoch [15/30] Loss=1.8968 Recon=1.8873 KL=0.9458
Epoch [16/30] Loss=1.6601 Recon=1.6498 KL=1.0288
Epoch [17/30] Loss=1.2030 Recon=1.1919 KL=1.1077
Epoch [18/30] Loss=0.7864 Recon=0.7746 KL=1.1817
Epoch [19/30] Loss=0.4814 Recon=0.4691 KL=1.2349
Epoch [20/30] Loss=0.2856 Recon=0.2730 KL=1.2555
Epoch [21/30] Loss=0.1760 Rec

In [16]:
vae.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        x = batch[0].to(device)

        logits, _, _ = vae(x)

        pred = logits.argmax(dim=2)

        target = x[:,1:]

        mask = (target != 0)

        correct += (
            ((pred == target) & mask)
            .sum()
            .item()
        )

        total += (
            mask
            .sum()
            .item()
        )

token_acc = correct / total

print(
    f"Token Accuracy: {token_acc:.4f}"
)

Token Accuracy: 0.9988


In [17]:
import torch

torch.save(
    vae.state_dict(),
    "transformer_vae_len60.pt"
)

print("Long model saved")

Long model saved
